# PyTorch 基本用法

## 1. 张量的创建与使用

张量（Tensor）是 PyTorch 中最基本的数据结构，类似于 NumPy 的数组，但可以在 GPU 上高效运行。

### 创建张量

In [ ]:
import torch

# 创建一个未初始化的张量
x = torch.empty(3, 3)  # 3x3 的张量
print(x)

# 创建一个随机初始化的张量
x = torch.rand(3, 3)  # 从均匀分布中随机采样
print(x)

# 创建一个零张量
x = torch.zeros(3, 3, dtype=torch.long)  # 指定数据类型为 long,long 类型通常用于存储较大范围的整数值,通常能够存储比 int 类型更大的整数值。
print(x)



In [ ]:
# 从 Python 列表创建张量
x = torch.tensor([1, 2, 3, 4])
print(x)

### 张量操作

In [ ]:
# 张量相加
y = torch.rand(3, 3)
z = x + y
print(z)


In [ ]:
# 张量切片
print(x[:, 1])  # 获取第一列

In [ ]:
# 张量形状操作
x = torch.rand(4, 4)
print(x)
print(x.view(16))  # 将张量重塑为一维
print(x.view(-1, 8))  # 重塑为 2x8

## 2. 数据创建和数据加载

PyTorch 提供了 torch.utils.data.Dataset 和 torch.utils.data.DataLoader 来方便地加载和处理数据。

1. torch.utils.data.Dataset
    Dataset是一个抽象类，用于存储数据和标签。它需要用户自己定义，通常包括以下两个主要方法：
    __len__：返回数据集中的样本数量。
    __getitem__：根据索引获取数据集中的单个样本。
    此外，Dataset类还可以包含其他方法和属性，如数据预处理、多任务学习等。用户需要根据自己的数据格式和需求来实现这个类。
    Dataset的主要目的是提供一个统一的接口来访问数据集中的样本，使得数据的加载和处理更加灵活和方便。
2. torch.utils.data.DataLoader
    DataLoader是一个迭代器，它封装了Dataset对象，并提供了一系列功能来方便地加载数据，包括：
    批量加载：将多个样本组合成一个批次进行加载，可以显著提高数据加载的效率。
    多进程加载：使用多个进程并行加载数据，进一步提高数据加载的速度。
    打乱数据：在每个epoch开始时随机打乱数据，有助于模型的训练。
    采样：支持自定义采样策略，如随机采样、顺序采样等。
    数据预处理：可以在DataLoader中进行数据预处理，如数据增强、标准化等。
    DataLoader的主要目的是提供一个高效、灵活的数据加载机制，使得模型训练过程中的数据加载更加高效和方便。
总结
    Dataset：负责存储和组织数据，提供统一的接口来访问数据集中的样本。
    DataLoader：负责从Dataset中高效地加载数据，提供批量加载、多进程加载、打乱数据等功能。
    在实际使用中，通常先定义一个Dataset类来存储和组织数据，然后创建一个DataLoader实例来加载这个Dataset中的数据。这样可以使得数据的加载和处理更加灵活和高效。

自定义数据集

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [ ]:
# 示例数据
data = torch.randn(10, 3, 32, 32)  # 10 张 3x32x32 的图像
labels = torch.randint(0, 10, (10,))  # 10 个标签

dataset = CustomDataset(data, labels)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)
    # 这行代码创建了一个DataLoader实例，其作用和参数解释如下：
    # dataset：这是传入DataLoader的参数，它应该是一个实现了torch.utils.data.Dataset接口的实例。这个实例包含了数据集的所有信息，包括数据的加载方式和数据集的大小。
    # batch_size=2：这个参数指定了每个批次（batch）中样本的数量。在这个例子中，每个批次将包含2个样本。这意味着当您迭代DataLoader时，每次迭代将返回2个样本。
    # shuffle=True：这个参数指定了是否在每个epoch开始时打乱数据集。在这里，设置为True表示在每个epoch开始时，数据集中的样本将被随机打乱。这有助于模型训练时的泛化能力，因为它确保了模型不会记住特定顺序的样本。
    # 综上所述，这行代码的作用是创建一个数据加载器，它将从指定的数据集中以批量大小为2的方式加载数据，并且在每个训练周期（epoch）开始时打乱数据集的顺序。这样的设置有助于提高模型的训练效果和泛化能力。


In [ ]:
for batch in dataloader:
    print(batch)

## 3. 数据增强

数据增强是通过变换输入数据来增加模型的泛化能力。PyTorch 的 torchvision.transforms 提供了许多数据增强方法。

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # 随机水平翻转
    transforms.RandomRotation(10),      # 随机旋转
    transforms.ToTensor()               # 转换为张量
])

# 应用到数据集
from torchvision.datasets import CIFAR10

dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

## 4. 网络模型创建

PyTorch 提供了 torch.nn.Module 来构建神经网络。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(784, 128)  # 28x28 输入，128 个隐藏单元
        self.fc2 = nn.Linear(128, 10)   # 输出 10 类

    def forward(self, x):
        x = x.view(-1, 784)  # 展平输入
        x = F.relu(self.fc1(x))  # 激活函数
        x = self.fc2(x)
        return x

model = SimpleNet()
print(model)

## 5. 使用 torch.autograd 自动求梯度

torch.autograd 是 PyTorch 的自动微分引擎，用于计算梯度。

In [ ]:
import torch

x = torch.randn(3, 3, requires_grad=True)  # 需要计算梯度
y = x * x
print(y)

y.sum().backward()  # 反向传播
print(x.grad)  # 输出梯度

## 6. 模型参数优化

优化器用于更新模型参数，以最小化损失函数。

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()  # 损失函数
optimizer = optim.SGD(model.parameters(), lr=0.01)  # 随机梯度下降

# 训练循环
for epoch in range(5):
    for inputs, labels in dataloader:
        optimizer.zero_grad()  # 清空梯度
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()  # 反向传播
        optimizer.step()  # 更新参数

## 7. 模型加载与保存

保存和加载模型是训练过程中的重要步骤。

### 保存模型

In [ ]:
# 保存整个模型
torch.save(model, 'model.pth')

# 仅保存模型参数
torch.save(model.state_dict(), 'model_state_dict.pth')

In [ ]:
## 加载模型

In [ ]:
# 加载整个模型
model = torch.load('model.pth')

# 加载模型参数
model = SimpleNet()
model.load_state_dict(torch.load('model_state_dict.pth'))